In [ ]:
# (DO NOT containerize this cell)
# Data handling parameters
param_minio_endpoint = 'scruffy.lab.uvalight.net:9000'
param_minio_virtual_lab_bucket = 'naa-vre-laserfarm'
param_files_to_process_file = 'AHN6_url_veluwe.txt'
param_max_tiles = 4
param_max_nodes = 2
param_original = 'original'

# Laserfarm parameters

In [ ]:
# Secrets (DO NOT containerize this cell)
from SecretsProvider import SecretsProvider
from getpass import getpass

secrets_provider = SecretsProvider(input_func=getpass)
secret_minio_access_key = secrets_provider.get_secret('secret_minio_access_key')
secret_minio_secret_key = secrets_provider.get_secret('secret_minio_secret_key')

In [ ]:
# (DO NOT containerize this cell)
import random
import math
import warnings
import requests

from minio import Minio
from minio.error import S3Error
from typing import List
from urllib import request
from pathlib import Path

In [ ]:
# (DO NOT containerize this cell)

conf_minio_path_to_process = 'toProcess'

In [ ]:
# datafiles processing manager
def get_filename_batches_to_process() -> List[List[str]]:
    filenames_to_process = get_minio_file_as_set(f"{conf_minio_path_to_process}/{param_files_to_process_file}", fail_if_file_not_found=True)
    if not filenames_to_process:
        raise ValueError(f"Found no files to process.")
    unprocessed_filenames = keep_unprocessed_urls(filenames_to_process)
    if not unprocessed_filenames:
        raise ValueError(f"The list of unprocessed_filenames is empty. Found {len(filenames_to_process)} files which have alreay been processed.")
    random.shuffle(unprocessed_filenames) 
    selected_filenames = unprocessed_filenames[:param_max_tiles]
    filename_batches = split_into_batches(selected_filenames)
    print(f"Files to process: {len(filenames_to_process)}, of which {len(unprocessed_filenames)} are unprocessed. Preparing {len(selected_filenames)} files in {len(filename_batches)} batches.")
    return filename_batches

def get_minio_file_as_set(filepath: str, fail_if_file_not_found: bool) -> set[str]:
    response = None
    try:
        response = minio_client.get_object(param_minio_virtual_lab_bucket, filepath)
        content = response.data.decode("utf-8")
        return {line.strip() for line in content.splitlines() if line.strip()}

    except S3Error as e:
        if ((not fail_if_file_not_found) and e.code == "NoSuchKey"):
            warnings.warn(f"No file found in path: {filepath}.")
            return set()
        raise e
        
    finally:
        if response:
            response.close()
            response.release_conn()
    
def keep_unprocessed_urls(urls: List[str]) -> List[str]:
    objects = minio_client.list_objects(param_minio_virtual_lab_bucket, prefix=param_original, recursive=True)
    processed_filenames = {str(Path(obj.object_name).name) for obj in objects}
    files_to_process = [url for url in urls if Path(url).name not in processed_filenames]
    return files_to_process

def split_into_batches(filenames: List[str]):
    batch_size = math.ceil(len(filenames)/param_max_nodes)
    return[filenames[i:i + batch_size] for i in range(0, len(filenames), batch_size)]

minio_client = Minio(
    param_minio_endpoint, 
    access_key=secret_minio_access_key,
    secret_key=secret_minio_secret_key,
    secure=True
)

las_data_filename_batches: List[List[str]] = get_filename_batches_to_process()

In [ ]:
# Retrieve laz files
def fetch_and_upload(url: str) -> None:
    with requests.get(url, stream=True) as response:
        response.raise_for_status() 
        file_size = int(response.headers.get('Content-Length', 0))
        filepath = Path(param_original) / Path(url).name
        minio_client.put_object(bucket_name=param_minio_virtual_lab_bucket, object_name=str(filepath), data=response.raw, length=file_size)
        
minio_client = Minio(
    param_minio_endpoint, 
    access_key=secret_minio_access_key,
    secret_key=secret_minio_secret_key,
    secure=True
)
    
for batch in las_data_filename_batches: 
    for url in batch:
        fetch_and_upload(url)

In [ ]:
# (DO NOT containerize this cell)
# Dummy use output
